# Prepare eukaryotic tool outputs

Clean functional-annotation outputs against exact reference `RNA_ID` values, export reproducible tables, and compare RNA-level coverage. The all-hit and significant-hit Kofam variants are retained for overlap analysis; only significant Kofam annotations enter the downstream functional table.

In [ ]:
from pathlib import Path

import pandas as pd

from benchannot.eukaryotic.prepare_output_tool import (
    TOOL_ORDER,
    build_cleaning_summary,
    build_functional_table,
    build_rna_presence,
    plot_cleaning_summary,
    plot_rna_upset,
    prepare_eggnog,
    prepare_interproscan,
    prepare_kofam,
    prepare_pannzer,
    prepare_reference,
    read_fasta_ids,
    save_table,
)

In [ ]:
cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (path for path in (cwd, *cwd.parents) if (path / "pyproject.toml").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate the BenchAnnot project root")

DATA_DIR = PROJECT_ROOT / "data" / "origin" / "eukaryote_output_tools"
EUKARYOTIC_OUTPUT = PROJECT_ROOT / "2_run" / "output" / "eukaryotic"
OUTPUT_DIR = EUKARYOTIC_OUTPUT / "tool_preparation"
TABLE_DIR = OUTPUT_DIR / "tables"
PLOT_DIR = OUTPUT_DIR / "plots"
TABLE_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

ORGANISMS = [
    {
        "organism": "saccharomyces_cerevisiae",
        "display_name": "Saccharomyces cerevisiae",
        "mitochondrial_prefix": "rna-Q",
    },
    {
        "organism": "drosophila_melanogaster",
        "display_name": "Drosophila melanogaster",
        "mitochondrial_prefix": "rna-Dmel",
    },
]
for config in ORGANISMS:
    organism = config["organism"]
    config["reference"] = (
        EUKARYOTIC_OUTPUT
        / "audit"
        / "reference"
        / organism
        / "prepared_reference"
        / "transcript_reference.tsv"
    )
    config["kofam"] = DATA_DIR / "kofamscan" / f"{organism}.kofam.txt"
    config["pannzer"] = DATA_DIR / "pannzer" / f"{organism}_pannzer.txt"
    config["eggnog"] = DATA_DIR / "eggnog" / f"{organism}_eggnog.emapper.annotations"
    config["interproscan"] = DATA_DIR / "interproscan" / f"{organism}.interpro.tsv"
    config["gffread_faa"] = DATA_DIR / "gffread" / f"{organism}_gffread.faa"

print(f"Tables: {TABLE_DIR}")
print(f"Plots:  {PLOT_DIR}")

## Clean and export tool tables

Coverage counts every returned parser row matched to the reference, including rows with an empty, missing, or placeholder description. Identifiers are compared exactly without trimming or version removal.

In [ ]:
references = {}
submitted_ids = {}
tool_results = {}
summary_parts = []

for config in ORGANISMS:
    organism = config["organism"]
    prefix = config["mitochondrial_prefix"]
    reference = prepare_reference(config["reference"])
    submitted_ids[organism] = {
        rna_id
        for rna_id in read_fasta_ids(config["gffread_faa"])
        if not rna_id.startswith(prefix)
    }
    kofam_all = prepare_kofam(config["kofam"], False, prefix)
    kofam_significant = prepare_kofam(config["kofam"], True, prefix)
    tools = {
        "Kofam": kofam_significant,
        "Pannzer": prepare_pannzer(config["pannzer"], prefix),
        "EggNOG": prepare_eggnog(config["eggnog"], prefix),
        "InterProScan": prepare_interproscan(config["interproscan"], prefix),
    }
    references[organism] = reference
    tool_results[organism] = {
        "Kofam_all": kofam_all,
        "Kofam_significant": kofam_significant,
        **{source: tools[source] for source in TOOL_ORDER[1:]},
    }

    save_table(reference, TABLE_DIR / f"{organism}_reference.tsv")
    save_table(kofam_all, TABLE_DIR / f"{organism}_kofam_all.tsv")
    save_table(kofam_significant, TABLE_DIR / f"{organism}_kofam_significant.tsv")
    for source in TOOL_ORDER[1:]:
        save_table(tools[source], TABLE_DIR / f"{organism}_{source.lower()}.tsv")
    functional = build_functional_table(reference, tools)
    save_table(functional, TABLE_DIR / f"{organism}_functional_annotations.tsv")
    summary_parts.append(
        build_cleaning_summary(
            organism, config["display_name"], reference, tools, submitted_ids[organism]
        )
    )
    print(
        f"{organism}: {len(reference):,} reference RNA_ID; "
        f"{len(functional):,} downstream rows"
    )

cleaning_summary = pd.concat(summary_parts, ignore_index=True)
save_table(cleaning_summary, TABLE_DIR / "cleaning_summary.tsv")
print(cleaning_summary.to_string(index=False))

## RNA_ID cleaning summary

Tools are rows and organisms are columns. Each panel has its own count scale because each tool emits and filters evidence differently. The three bars report exact GFFread input `RNA_ID`, tool-identified `RNA_ID` after mitochondrial removal, and `RNA_ID` retained after tool-specific selection and exact reference matching.

In [ ]:
print(
    cleaning_summary[
        ["display_name", "source", "gffread_input_rna_ids", "tool_identified_rna_ids", "retained_rna_ids", "represented_loci"]
    ].to_string(index=False)
)
plot_cleaning_summary(cleaning_summary, PLOT_DIR / "cleaning_summary.png")
print(f"Saved {PLOT_DIR / 'cleaning_summary.png'}")

## Exact RNA_ID intersections

The UpSet element universe is the exact prepared-reference `RNA_ID`, not a locus-level collapse. Both all-hit and significant-hit Kofam comparisons are shown. Categories are displayed by set cardinality, from the largest set to the smallest; exported membership tables retain the fixed source order Reference, Kofam, Pannzer, EggNOG, InterProScan.

In [ ]:
for config in ORGANISMS:
    organism = config["organism"]
    reference = references[organism]
    common_tools = {
        source: tool_results[organism][source] for source in TOOL_ORDER[1:]
    }
    display_math = config["display_name"].replace(" ", r"\ ")
    for variant, kofam_label, kofam_table in (
        ("all_kofam", "all Kofam hits", tool_results[organism]["Kofam_all"]),
        ("significant_kofam", "significant Kofam hits", tool_results[organism]["Kofam_significant"]),
    ):
        tools = {"Kofam": kofam_table, **common_tools}
        presence, membership = build_rna_presence(reference, tools)
        save_table(presence, TABLE_DIR / f"{organism}_presence_{variant}.tsv")
        save_table(
            membership.reset_index(),
            TABLE_DIR / f"{organism}_membership_{variant}.tsv",
        )
        title = f"$\it{{{display_math}}}$ IDs similarity"
        print(f"\n{config['display_name']} - {kofam_label}")
        print(membership.sum().to_string())
        plot_rna_upset(
            membership, title, PLOT_DIR / f"{organism}_upset_{variant}.png"
        )

## Drosophila first-reference-transcript intersections

Drosophila has multiple transcripts per locus, so a second UpSet analysis uses the first transcript in reference table order as one deterministic representative per `locus_tag`. Tool memberships are restricted to those representative `RNA_ID` values. This is a noncanonical locus-level sensitivity view; it does not replace the all-RNA_ID analysis.

In [ ]:
from benchannot.eukaryotic.functional_analysis import select_first_reference_transcript

config = next(item for item in ORGANISMS if item["organism"] == "drosophila_melanogaster")
organism = config["organism"]
reference = select_first_reference_transcript(references[organism])
representative_ids = set(reference["RNA_ID"])
display_math = config["display_name"].replace(" ", r"\ ")
print(f"{config['display_name']} first-reference transcript representatives: {len(reference):,} RNA_ID; {reference['locus_tag'].nunique():,} loci")

for variant, kofam_label, kofam_table in (
    ("all_kofam", "all Kofam hits", tool_results[organism]["Kofam_all"]),
    ("significant_kofam", "significant Kofam hits", tool_results[organism]["Kofam_significant"]),
):
    tools = {"Kofam": kofam_table, **{source: tool_results[organism][source] for source in TOOL_ORDER[1:]}}
    representative_tools = {
        source: table.loc[table["RNA_ID"].isin(representative_ids)].copy()
        for source, table in tools.items()
    }
    presence, membership = build_rna_presence(reference, representative_tools)
    save_table(presence, TABLE_DIR / f"{organism}_presence_first_reference_transcript_{variant}.tsv")
    save_table(
        membership.reset_index(),
        TABLE_DIR / f"{organism}_membership_first_reference_transcript_{variant}.tsv",
    )
    title = f"$\it{{{display_math}}}$ IDs similarity"
    print(f"\n{config['display_name']} first-reference transcript - {kofam_label}")
    print(membership.sum().to_string())
    plot_rna_upset(
        membership,
        title,
        PLOT_DIR / f"{organism}_upset_first_reference_transcript_{variant}.png",
    )